# Pan-cancer model training

In [1]:
import time
import pandas as pd
from monte import train_with_cv

## Load data

**Training set**

In [ ]:
# the parquet file and metadata csv file could be downloaded from Zenodo
df_beta_train = pd.read_parquet("../../data/methylation/train_val_pan-cancer_beta.parquet")
df_meta_train = pd.read_csv("../../data/methylation/train_val_pan-cancer_meta.csv")
df_meta_train = df_meta_train.set_index("Barcode", drop=False)
df_meta_train = df_meta_train.loc[df_beta_train.index]

In [3]:
trained_metric = "CPE"
df_meta_metric = df_meta_train.dropna(subset=[trained_metric])
df_beta_metric = df_beta_train.loc[df_meta_metric["Barcode"]]

**Test set**

In [ ]:
# the parquet file and metadata csv file could be downloaded from Zenodo
df_beta_test = pd.read_parquet("../../data/methylation/test_pan-cancer_beta.parquet")
df_meta_test = pd.read_csv("../../data/methylation/test_pan-cancer_meta.csv")
df_meta_test = df_meta_test.set_index("Barcode", drop=False)
df_meta_test = df_meta_test.loc[df_beta_test.index]

## Model training and prediction

Training pan-cancer monte model

In [5]:
start = time.time()
monte = train_with_cv(df_beta_metric, df_meta_metric[trained_metric])
end = time.time()
training_time = end - start

In [6]:
monte.best_top_n

250

In [7]:
df_meta_train[f"predicted_{trained_metric}"] = monte.predict_purity(df_beta_train)
df_meta_test[f"predicted_{trained_metric}"] = monte.predict_purity(df_beta_test)

For timing purpose only

In [8]:
cancer_types = df_meta_test["Cancer.type"].unique()
df_cancer_pred_time = pd.DataFrame({
    "cancer": cancer_types,
    "prediction_time": [0.0] * len(cancer_types),
    "n_samples": [0] * len(cancer_types)
})

for i, cancer in enumerate(cancer_types):
    df_cancer = df_meta_test[df_meta_test["Cancer.type"] == cancer]
    df_beta_cancer = df_beta_test.loc[df_cancer["Barcode"]]
    
    start = time.time()
    monte.predict_purity(df_beta_cancer)
    end = time.time()
    
    prediction_time = end - start
    df_cancer_pred_time.loc[i, "prediction_time"] = prediction_time
    df_cancer_pred_time.loc[i, "n_samples"] = len(df_cancer)

## Save model and results

In [9]:
monte.save(f"../../data/monte_outputs/trained_models/monte_pancancer_model.pkl")

In [10]:
df_meta_train.to_csv(f"../../data/monte_outputs/pancancer/monte_pancancer_meta_with_predictions_train.csv", index=False)
df_meta_test.to_csv(f"../../data/monte_outputs/pancancer/monte_pancancer_meta_with_predictions_test.csv", index=False)

In [11]:
with open("../../data/monte_outputs/pancancer/monte_pancancer_training_time.csv", "w") as f:
    f.write("Method,Training Time (seconds),n_samples\n")
    f.write(f"MONTE,{training_time},{len(df_beta_metric)}\n")

In [12]:
df_cancer_pred_time.to_csv("../../data/monte_outputs/pancancer/monte_pancancer_prediction_time_by_cancer.csv", index=False)